In [17]:
from __future__ import annotations

from collections.abc import Sequence
from typing import TypedDict

import random

from gymnasium.spaces import Box
from gymnasium.spaces import Dict
from gymnasium.spaces import Discrete
from pettingzoo.utils import AECEnv
from pettingzoo.utils import agent_selector

from collections import deque

from torch import nn
from torch import optim

import torch
import torch.nn.functional as F

from tqdm import tqdm

import numpy as np

In [18]:
# ----------------------------------------------------
# Вспомогательный TypedDict для удобства типизации
# ----------------------------------------------------
class DurakObservation(TypedDict):
    hand: list[int]
    cards_on_table: list[int]
    cards_in_discard: list[int]
    trump_suit: int
    is_attacker: bool
    opp_cards_count: int
    deck_count: int
    attacking_card: int | None

In [19]:
# ----------------------------------------------------
# Константы
# ----------------------------------------------------
NUM_PLAYERS = 2
DECK_SIZE = 36
ALL_AGENTS = [f'player_{i}' for i in range(NUM_PLAYERS)]

In [20]:
# Ранги и масти (всего 36 карт)
# card_id: 0..35
# rank = card_id // 4  (0..8)    # 9 "старших" рангов: 6,7,8,9,10,В,Д,К,Т(туз)
# suit = card_id % 4   (0..3)    # 0=hearts,1=diamonds,2=clubs,3=spades
def get_rank(card_id: int) -> int:
    return card_id // 4


def get_suit(card_id: int) -> int:
    return card_id % 4


def card_can_beat(
    defend_card: int, trump_suit: int, attacking_card: int
) -> bool:
    """Проверяем, может ли карта `defend_card` побить `attacking_card` при условии масти `trump_suit`."""
    if attacking_card is None:
        return True  # Если нет атакующей карты, считаем True (хотя логика странная)

    attack_rank = get_rank(attacking_card)
    attack_suit = get_suit(attacking_card)

    defend_rank = get_rank(defend_card)
    defend_suit = get_suit(defend_card)

    # 1) Если та же масть, но ранг выше
    if defend_suit == attack_suit and defend_rank > attack_rank:
        return True

    # 2) Если defend — козырь, а attack — нет
    if defend_suit == trump_suit and attack_suit != trump_suit:
        return True

    # 3) Если оба козыря, но ранг defend выше
    if defend_suit == trump_suit and attack_suit == trump_suit:
        return defend_rank > attack_rank

    return False


def is_rank_in_play(card_id: int, cards_on_table: list[int]) -> bool:
    """Проверяем, есть ли на столе карта такого же ранга (для подбрасывания)."""
    if not cards_on_table:
        return True  # если стол пуст, можно ходить любой картой
    table_ranks = {get_rank(c) for c in cards_on_table}
    return get_rank(card_id) in table_ranks

In [21]:
# ----------------------------------------------------
# Класс окружения
# ----------------------------------------------------
class DurakAEC(AECEnv):
    """
    Пример среды "Дурак" для двух игроков, с упрощённой логикой многократной атаки:
    - Атакующий может подкидывать карты (те, чей ранг есть на столе).
    - Защитник может побить или взять карты.
    - Если защитник взял карты, раунд заканчивается немедленно,
      и в следующем раунде снова ходит тот же атакующий.
    - Если атакующий сказал "бито" (37), раунд заканчивается, и роли меняются.
    - Игра завершается, когда у одного из игроков нет карт в руке и колода пуста:
      этот игрок побеждает, второй проигрывает.
    """

    metadata = {'render_modes': ['human'], 'name': 'durak_multi_attack_v0'}

    def __init__(self, seed: int | None = None, agents: Sequence = ()):
        """
        :param seed: для воспроизводимости
        :param agents: список внешних агентов (можете использовать для self-play или нет).
                       Здесь просто сохраним их, а в демо-коде ниже покажем, как это может выглядеть.
        """
        super().__init__()
        self.seed_value = seed
        self.external_agents = list(agents)  # Сохраняем агентов (при желании)

        self.possible_agents = ALL_AGENTS
        self.agent_name_mapping = {
            name: i for i, name in enumerate(self.possible_agents)
        }

        # Действия: 0..35 (сыграть карту), 36 = "взять", 37 = "бито"
        self.action_spaces = {
            agent: Discrete(DECK_SIZE + 2)  # 38
            for agent in self.possible_agents
        }

        # Наблюдение: используем Dict, чтобы отражать структуру
        # (для упрощения некоторые поля закодируем как Box или Discrete).
        # В реальности можно это уточнить.
        self.observation_spaces = {}
        for agent in self.possible_agents:
            # hand, cards_on_table, cards_in_discard — списки переменной длины,
            # поэтому строго описать MultiDiscrete сложновато.
            # Для демонстрации укажем просто Box(...) c некоторым upper_bound.
            # Или, как вариант, можно было бы ограничиться max размером руки (36).
            obs_dict = Dict({
                'hand': Box(
                    low=0, high=DECK_SIZE, shape=(36,), dtype=np.int32
                ),  # упрощённо
                'cards_on_table': Box(
                    low=0, high=DECK_SIZE, shape=(36,), dtype=np.int32
                ),
                'cards_in_discard': Box(
                    low=0, high=DECK_SIZE, shape=(36,), dtype=np.int32
                ),
                'trump_suit': Discrete(4),
                'is_attacker': Discrete(2),
                'opp_cards_count': Box(
                    low=0, high=36, shape=(), dtype=np.int32
                ),
                'deck_count': Box(low=0, high=36, shape=(), dtype=np.int32),
                'attacking_card': Box(
                    low=0, high=DECK_SIZE, shape=(), dtype=np.int32
                ),
            })
            self.observation_spaces[agent] = obs_dict

        self.reset()

    def seed(self, seed=None):
        """Устанавливаем seed для рандома."""
        if seed is not None:
            self.seed_value = seed
        random.seed(self.seed_value)
        np.random.seed(self.seed_value)

    def reset(self, seed=None, return_info=False, options=None):
        self.seed(seed if seed is not None else self.seed_value)

        self.agents = self.possible_agents[:]
        self._agent_selector = agent_selector(self.agents)
        self.agent_selection = self._agent_selector.reset()

        # Перемешиваем колоду
        self.deck = list(range(DECK_SIZE))
        random.shuffle(self.deck)

        # Руки
        self.hands = {agent: [] for agent in self.agents}
        self.cards_on_table = []  # Карты на столе (упростим: будем класть все атакующие карты + бьющие?)
        self.cards_in_discard = []  # Бито/сброс

        self.trump_suit = None
        self.attacking_card = (
            None  # текущая атакующая карта (если идёт защита прямо сейчас)
        )

        # bool: True => агент является атакующим
        self.is_attacker = dict.fromkeys(self.agents, False)

        # Флаги окончания
        self.terminations = dict.fromkeys(self.agents, False)
        self.truncations = dict.fromkeys(self.agents, False)
        self.rewards = dict.fromkeys(self.agents, 0.0)
        self.infos = {agent: {} for agent in self.agents}

        self.game_started = False

        if return_info:
            return {}, {}

    def start(self):
        """Старт игры: раздаём по 6 карт, определяем козырь, выбираем первого атакующего."""
        for i in range(6):
            for agent in self.agents:
                if self.deck:
                    self.hands[agent].append(self.deck.pop())

        # Верхняя карта колоды задаёт козырь:
        if self.deck:
            top_card = self.deck[-1]
            self.trump_suit = get_suit(top_card)
        else:
            self.trump_suit = 0  # если вдруг колода пуста

        # Определяем, у кого минимальная козырная карта
        min_trump_card_per_agent = {}
        for agent in self.agents:
            trumps = [
                c for c in self.hands[agent] if get_suit(c) == self.trump_suit
            ]
            if trumps:
                min_trump_card_per_agent[agent] = min(trumps)
            else:
                min_trump_card_per_agent[agent] = None

        # Если у обоих None => ходит второй
        if (
            min_trump_card_per_agent[self.agents[0]] is None
            and min_trump_card_per_agent[self.agents[1]] is None
        ):
            first_agent = self.agents[1]
        else:
            # Выбираем агента с минимальным козырём
            not_none = [
                (a, c)
                for a, c in min_trump_card_per_agent.items()
                if c is not None
            ]
            not_none.sort(key=lambda x: x[1])
            first_agent = not_none[0][0] if not_none else self.agents[1]

        self.is_attacker[first_agent] = True

        # Определим порядок в agent_selector так, чтобы всегда:
        #   attacker -> defender -> attacker -> defender ...
        # до тех пор, пока раунд не завершится (тогда сделаем дополнительные действия).
        # Но в AEC обычно мы просто используем round-robin. Нам придётся "ручками" иногда
        # не переводить ход дальше, если хотим "несколько ходов атакующего подряд".
        self._agent_selector = agent_selector(
            [first_agent] + [a for a in self.agents if a != first_agent]
        )
        self.agent_selection = self._agent_selector.reset()

        self.game_started = True

    def observe(self, agent):
        return self._make_observation(agent)

    def last(self, observe=True):
        if not self.game_started:
            return None
        agent = self.agent_selection
        if observe:
            return self._make_observation(agent)
        return None

    def _make_observation(self, agent) -> DurakObservation:
        opp_agent = [a for a in self.agents if a != agent][0]

        obs: DurakObservation = {
            'hand': sorted(self.hands[agent]),
            'cards_on_table': sorted(self.cards_on_table),
            'cards_in_discard': sorted(self.cards_in_discard),
            'trump_suit': self.trump_suit,
            'is_attacker': self.is_attacker[agent],
            'opp_cards_count': len(self.hands[opp_agent]),
            'deck_count': len(self.deck),
            'attacking_card': self.attacking_card,
        }
        return obs

    def step(self, action):
        """Одновременная логика "многоходовой" атаки:
        - Если ход атакующего: сыграть карту (подкидываем) или 'бито' (37).
        - Если ход защитника: побить карту (если может) или 'взять' (36).
        """
        if not self.game_started:
            raise ValueError('Игра не начата, вызовите start()')

        agent = self.agent_selection
        if self.terminations[agent] or self.truncations[agent]:
            self._finish_step()
            return

        opp_agent = [a for a in self.agents if a != agent][0]

        # Действие:
        # 0..35 => сыграть карту
        # 36 => взять
        # 37 => бито (пас от атакующего)
        if self.is_attacker[agent]:
            # Ход атакующего
            if action < 36:
                # Проверяем, есть ли карта в руке
                if action in self.hands[agent]:
                    # Проверим, можем ли мы "подкинуть" эту карту
                    # Упрощённая логика: разрешаем ходить любой картой, если стол пуст
                    # или ранг есть на столе, если не пуст.
                    if is_rank_in_play(action, self.cards_on_table):
                        # Атакуем
                        self.hands[agent].remove(action)
                        self.cards_on_table.append(action)
                        # Теперь атакующая карта = последняя
                        self.attacking_card = action
                        # Передаём ход защитнику
                        self._finish_step(force_next=opp_agent)
                    else:
                        # Недопустимый ход. Можно выдать штраф или проигнорировать.
                        # Для упрощения — просто игнорируем.
                        self._finish_step()
                else:
                    self._finish_step()
            elif action == 36:
                # 36="взять" для атакующего не имеет смысла, игнорируем
                self._finish_step()
            elif action == 37:
                # "Бито" — атакующий говорит, что подкидывать больше не будет,
                # значит раунд закончился => переносим все карты со стола в discard,
                # меняем роли
                self._round_end(defender_took=False)
                self._finish_step()
            else:
                self._finish_step()
        # Ход защитника
        elif action < 36:
            # Пытаемся побить атакующую карту
            if action in self.hands[agent] and self.attacking_card is not None:
                can_beat_it = card_can_beat(
                    action, self.trump_suit, self.attacking_card
                )
                if can_beat_it:
                    # Успешно бьём: убираем карту из руки
                    self.hands[agent].remove(action)
                    # Ставим и атакующую, и бьющую в discard
                    self.cards_in_discard.append(self.attacking_card)
                    self.cards_in_discard.append(action)
                    # Убираем их со стола
                    if self.attacking_card in self.cards_on_table:
                        self.cards_on_table.remove(self.attacking_card)
                    self.attacking_card = None
                    # Возвращаем ход атакующему, он может подкинуть ещё
                    self._finish_step(force_next=opp_agent)
                else:
                    # Нельзя побить — действие игнорируем
                    self._finish_step()
            else:
                self._finish_step()
        elif action == 36:
            # Защитник берёт карты
            # Забирает все карты со стола + attacking_card (если она там)
            self.hands[agent].extend(self.cards_on_table)
            if (self.attacking_card is not None) and (
                self.attacking_card not in self.cards_on_table
            ):
                # Если вдруг она уже убрана со стола, пропустим
                pass
            elif self.attacking_card is not None:
                self.hands[agent].append(self.attacking_card)

            self.cards_on_table.clear()
            self.attacking_card = None

            # Раунд закончен, роли не меняются (аттакующий снова ходит в следующем раунде)
            self._round_end(defender_took=True)
            self._finish_step()
        elif action == 37:
            # "Бито" от защитника не имеет смысла
            self._finish_step()
        else:
            self._finish_step()

        self._check_game_done()

    def _round_end(self, defender_took: bool):
        """Завершение раунда: добор карт, смена (или не смена) роли."""
        # Сначала добираем (атакующему, потом защищающемуся)
        order = sorted(self.agents, key=lambda a: not self.is_attacker[a])
        for a in order:
            while len(self.hands[a]) < 6 and self.deck:
                self.hands[a].append(self.deck.pop())

        if defender_took:
            # Защитник взял — атакующий остаётся атакующим
            pass
        else:
            # Защитник отбился => меняем роли
            for a in self.agents:
                self.is_attacker[a] = not self.is_attacker[a]

        # Всё, что осталось на столе (если вдруг), скидываем в discard
        self.cards_in_discard.extend(self.cards_on_table)
        self.cards_on_table.clear()
        self.attacking_card = None

    def _finish_step(self, force_next=None):
        """Переходим к следующему агенту (или к агенту из force_next, если хотим)."""
        if force_next is not None:
            self.agent_selection = force_next
        else:
            self.agent_selection = self._agent_selector.next()

        # Если следующий агент уже terminated, идём дальше
        while (
            self.terminations[self.agent_selection]
            or self.truncations[self.agent_selection]
        ) and self.agents:
            self.agent_selection = self._agent_selector.next()

    def _check_game_done(self):
        """Игра завершается, если все кроме одного вышли (или для 2 игроков:
        как только один выполнил условие 'нет карт + колода пуста', второй проигрывает).
        """
        # Ставим termination тому, у кого 0 карт и колода пуста => он выиграл
        for agent in self.agents:
            if len(self.hands[agent]) == 0 and len(self.deck) == 0:
                self.terminations[agent] = True
                # Можно дать награду победителю
                self.rewards[agent] = 1.0

        # Считаем, сколько не "закончили"
        active_agents = [a for a in self.agents if not self.terminations[a]]
        if len(active_agents) == 1:
            # Остался ровно один - он проигрывает
            loser = active_agents[0]
            self.terminations[loser] = True
            self.rewards[loser] = -1.0

            # Те, кто уже в терминации, должны иметь +1 если ещё не проставлено
            for ag in self.agents:
                if ag != loser and self.rewards[ag] == 0.0:
                    self.rewards[ag] = 1.0

    def render(self, mode='human'):
        if mode == 'human':
            agent = self.agent_selection
            print(f'--- Agent turn: {agent} ---')
            for ag in self.agents:
                print(
                    f'{ag} => hand: {self.hands[ag]}, attacker={self.is_attacker[ag]}, '
                    f'terminated={self.terminations[ag]}'
                )
            print(
                f'Cards on table: {self.cards_on_table}, Attacking card: {self.attacking_card}'
            )
            print(
                f'Discard: {self.cards_in_discard}, Deck: {len(self.deck)}, Trump suit: {self.trump_suit}'
            )
            print('---')

In [22]:
class SimpleDQN(nn.Module):
    """
    Очень простой MLP, принимающий "плоское" состояние
    и предсказывающий Q-значения для 38 действий.
    """

    def __init__(self, obs_dim: int, act_dim: int = 38):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.out = nn.Linear(128, act_dim)

    def forward(self, x):
        x = F.selu(self.fc1(x))
        x = F.selu(self.fc2(x))
        return self.out(x)

In [23]:
def flatten_observation(obs: DurakObservation, agent_id: int) -> np.ndarray:
    """
    Пример конвертации словаря наблюдений в плоский вектор.
    Включим туда:
      - one-hot: кто мы (agent_id)
      - наша рука (36 длина, 1/0 есть ли карта)
      - карты на столе (36 длина)
      - размер колоды, opp_cards_count
      - trump_suit (1 из 4)
      - is_attacker (0 или 1)
      - attacking_card (36 длина one-hot)
    """
    vec = []

    # one-hot код агента (2 игрока)
    agent_one_hot = [0, 0]
    agent_one_hot[agent_id] = 1
    vec.extend(agent_one_hot)

    # Наша рука
    hand_vec = [0] * DECK_SIZE
    for c in obs['hand']:
        hand_vec[c] = 1
    vec.extend(hand_vec)

    # Карты на столе
    table_vec = [0] * DECK_SIZE
    for c in obs['cards_on_table']:
        table_vec[c] = 1
    vec.extend(table_vec)

    # attacking_card - one-hot
    att_vec = [0] * DECK_SIZE
    if obs['attacking_card'] is not None:
        att_vec[obs['attacking_card']] = 1
    vec.extend(att_vec)

    # discard не будем явно включать (или можно включить)
    # trump_suit (4)
    ts = [0, 0, 0, 0]
    ts[obs['trump_suit']] = 1
    vec.extend(ts)

    # is_attacker
    vec.append(1.0 if obs['is_attacker'] else 0.0)

    # opp_cards_count + deck_count (2 штуки)
    vec.append(float(obs['opp_cards_count']))
    vec.append(float(obs['deck_count']))

    return np.array(vec, dtype=np.float32)

In [24]:
class ReplayBuffer:
    """Простой буфер опыта для DQN."""

    def __init__(self, capacity=10000) -> None:
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done) -> None:
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = [], [], [], [], []
        for i in indices:
            s, a, r, ns, d = self.buffer[i]
            states.append(s)
            actions.append(a)
            rewards.append(r)
            next_states.append(ns)
            dones.append(d)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)

In [27]:
def dqn_selfplay_training(num_episodes=2000, render=False) -> None:
    """
    Пример (упрощённый) цикла обучения DQN в среде DurakAEC.
    Один Q-network на двоих агентов. Мы кодируем "кто ходит" в состоянии.
    """
    env = DurakAEC()
    env.reset()
    env.start()

    # Определим размерность наблюдения:
    # По нашей функции flatten_observation:
    # 2 (agent_id one-hot) + 36 (hand) + 36 (table) + 36 (att_card) + 4 (trump_suit) + 1 (is_attacker)
    # + 2 (opp_cards_count, deck_count) = 117
    obs_dim = 117
    act_dim = 38

    policy_net = SimpleDQN(obs_dim, act_dim)
    target_net = SimpleDQN(obs_dim, act_dim)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
    replay_buffer = ReplayBuffer(capacity=50000)
    batch_size = 64
    gamma = 0.99
    epsilon = 1.0  # eps-greedy
    epsilon_decay = 0.9995
    min_epsilon = 0.05
    update_target_freq = 1000
    global_step = 0

    def select_action(state_vec):
        if random.random() < epsilon:
            return random.randint(0, act_dim - 1)
        with torch.no_grad():
            q_values = policy_net(
                torch.tensor(state_vec, dtype=torch.float32).unsqueeze(0)
            )
            return q_values.argmax(dim=1).item()

    for episode in tqdm(range(num_episodes), desc='episode'):
        env.reset()
        env.start()

        done = False
        episode_steps = 0

        # Словарь: agent -> его последнее состояние (для s,a,r перехода)
        last_obs = {}
        last_state_vec = {}

        while not done:
            agent = env.agent_selection

            if env.terminations[agent] or env.truncations[agent]:
                # Агент завершил — step(None) чтобы пропустить
                env.step(None)
            else:
                obs = env.last(observe=True)
                agent_id = env.agent_name_mapping[agent]
                state_vec = flatten_observation(obs, agent_id)

                action = select_action(state_vec)

                # Сохраняем (s) для последующего перехода
                last_obs[agent] = obs
                last_state_vec[agent] = state_vec

                env.step(action)

                # После шага мы можем собрать (s, a, r, s', done)
                reward = env.rewards[agent]
                # Определим done для этого агента как (termination or truncation)
                done_agent = env.terminations[agent] or env.truncations[agent]

                # new_obs = env.last(observe=True) -- но уже для другого агента,
                # поэтому чтобы собрать переход, мы используем "старое" состояние -> "текущее" (которое будет затем)
                # Упростим: когда ход агента закончился, можно взять next_state из перспективы этого же агента.
                # Но в AEC это tricky. Упростим и скажем, что s' = state_vec (agent) (хотя это не идеально).

                # Будем пушить сразу
                if done_agent:
                    # Если агент закончил — у него next_state = нули
                    ns_vec = np.zeros_like(state_vec)
                else:
                    # Можно либо подождать, когда снова будет ход этого агента (правильнее),
                    # либо (упрощённо) взять текущее состояние (для другого агента) как next_state.
                    # Для простоты запишем так:
                    ns_vec = state_vec

                replay_buffer.push(
                    last_state_vec[agent],
                    action,
                    reward,
                    ns_vec,
                    float(done_agent),
                )

                episode_steps += 1

                # Обновляем DQN, если в буфере достаточно переходов
                if len(replay_buffer) > batch_size:
                    states_b, actions_b, rewards_b, next_states_b, dones_b = (
                        replay_buffer.sample(batch_size)
                    )

                    states_t = torch.tensor(states_b, dtype=torch.float32)
                    actions_t = torch.tensor(
                        actions_b, dtype=torch.int64
                    ).unsqueeze(1)
                    rewards_t = torch.tensor(
                        rewards_b, dtype=torch.float32
                    ).unsqueeze(1)
                    next_states_t = torch.tensor(
                        next_states_b, dtype=torch.float32
                    )
                    dones_t = torch.tensor(
                        dones_b, dtype=torch.float32
                    ).unsqueeze(1)

                    # Q(s,a)
                    q_values = policy_net(states_t).gather(1, actions_t)

                    # Q'(s', a') = max_a' target_net(s')  (Double DQN делать не будем, упрощённо)
                    with torch.no_grad():
                        next_q = target_net(next_states_t).max(
                            dim=1, keepdim=True
                        )[0]

                    target = rewards_t + gamma * (1 - dones_t) * next_q
                    loss = F.mse_loss(q_values, target)

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    # Обновляем target_net
                    global_step += 1
                    if global_step % update_target_freq == 0:
                        target_net.load_state_dict(policy_net.state_dict())

                # decay eps
                epsilon = max(min_epsilon, epsilon * epsilon_decay)

            done = all(
                env.terminations[a] or env.truncations[a] for a in env.agents
            )
            if render:
                env.render()

        # Пример: выводим прогресс
        if (episode + 1) % 100 == 0:
            print(f'Episode {episode + 1}, epsilon={epsilon:.3f}')

    print('Training finished.')
    return target_net

In [28]:
dqn_selfplay_training(num_episodes=50, render=True)

episode: 100%|██████████| 50/50 [06:48<00:00,  8.16s/it]

Training finished.
